# Zepto Data Analysis

Visual analysis of the Zepto inventory data using Pandas, Matplotlib and Seaborn. This goes with the SQL analysis — same dataset, just making charts to see the patterns.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Load the data

Reading in the CSV and checking what it looks like.

In [ ]:
df = pd.read_csv('zepto_v2.csv')
# Rename Category to match standard naming
df = df.rename(columns={'Category': 'category'})
display(df.head())

## Cleaning up

Same steps as the SQL side — prices are in paise so we need to convert them to rupees, and drop anything with zero MRP.

In [ ]:
# Convert paise to rupees
df['mrp'] = df['mrp'] / 100.0
df['discountedSellingPrice'] = df['discountedSellingPrice'] / 100.0

# Remove products where MRP is 0
df = df[df['mrp'] > 0]

print(f"Dataset shape after cleaning: {df.shape}")

## Visualizations

### Category distribution

Which categories have the most products? Let's find out.

In [ ]:
top_categories = df['category'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(y=top_categories.index, x=top_categories.values, hue=top_categories.index, palette="viridis", legend=False)\nplt.title('Top 10 Product Categories by SKU Count', fontsize=14)
plt.xlabel('Number of Products')
plt.ylabel('Category')
plt.show()

### Discount distribution

Checking how discounts are spread across products.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['discountPercent'], bins=20, kde=True, color='coral')
plt.title('Distribution of Discount Percentages', fontsize=14)
plt.xlabel('Discount Percentage (%)')
plt.ylabel('Frequency')
plt.show()

### Out-of-stock by category

Seeing which categories have stock issues.

In [ ]:
# Filter for top 10 categories
top_10_cats = df['category'].value_counts().head(10).index
df_top10 = df[df['category'].isin(top_10_cats)]

# Calculate out of stock percentages
oos_stats = df_top10.groupby('category')['outOfStock'].value_counts(normalize=True).unstack(fill_value=0) * 100
oos_stats = oos_stats.rename(columns={False: 'In Stock (%)', True: 'Out of Stock (%)'})
oos_stats = oos_stats.sort_values('Out of Stock (%)', ascending=False)

oos_stats.plot(kind='bar', stacked=True, figsize=(12, 6), color=['#2ca02c', '#d62728'])
plt.title('Stock Availability by Top 10 Categories', fontsize=14)
plt.xlabel('Category')
plt.ylabel('Percentage (%)')
plt.legend(title='Status', loc='upper right')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Price vs discount

Scatter plot to see if higher priced items get better discounts. Filtered to MRP < 2000 to keep it readable.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df[df['mrp'] < 2000], x='mrp', y='discountPercent', alpha=0.5, color='purple')
plt.title('MRP vs Discount Percentage (Products < ₹2000)', fontsize=14)
plt.xlabel('MRP (₹)')
plt.ylabel('Discount (%)')
plt.show()